# LingGym reproduction: Hacker & Implementer

**Paper:** Yang et al. (2025), *LingGym: How Far Are LLMs from Thinking Like Field Linguists?* (EMNLP 2025)

Thanush Dinesh, CSCI3052U Machine Learning, individual assignment

LingGym turns example sentences from 18 grammar books into fill-in-the-blank questions: one word in a
glossed sentence is hidden, and the model picks the right word + gloss from 4 options. The model gets
more and more hints: **S** (sentence), **G** (gloss), **KP** (grammar explanation), **T** (translation).

**Before running in Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all (~40 min).

| Part | What happens | Time on T4 |
|---|---|---|
| 1. Data | load the authors' questions, check against the paper, pick our sample | 1 min |
| 2. Our question generator | rebuild questions from the raw grammar data (the "hacker" part) | 2 min |
| 3. Model + live demo | load Qwen2.5-7B, watch it answer one question with more and more hints | 5 min |
| 4. Experiments | Exp 0 random, Exp 1 released questions, Exp 2 our questions, Exp 3 paper-style sampling | ~30 min |
| 5. Results vs. paper | comparison table, chart, per-language breakdown, wrong-answer types | 1 min |
| 6. Conclusion | what the results mean, limitations | |

All runs save to `results/<run>/` after every batch. If Colab disconnects, re-run and it picks up where it stopped.

In [ ]:
# --- Setup: works in Google Colab and locally -------------------------------
import os, sys
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Thanush38/CSCI3052U-Machine-learning-individual-assignment.git"

REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        !git clone -q $REPO_URL $REPO_DIR
    %cd $REPO_DIR
    !pip install -q -r requirements-colab.txt
elif os.path.basename(os.getcwd()) == "notebook":
    os.chdir("..")  # run from the repo root so src/, data/ and results/ resolve
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
# --- Settings: everything you might want to change is here ------------------
SEED = 42                                          # fixes random, numpy and torch
LANGUAGES = ["Pichi", "Papuan_Malay", "Kalamang"]  # folder names in the LingGym repo
N_PER_LANGUAGE = 150                               # questions per language
MODEL = "Qwen/Qwen2.5-7B-Instruct"                 # same model as the paper's Table 3
QUANTIZATION = "4bit"                              # "4bit" fits a free T4; "none" = float16
RAW = "data/raw/LingGym"                           # where the authors' data is downloaded
DATA_COMMIT = "ce6059b47c8839ac3de1b976a858cfbfc6daa6c9"  # exact version of their data

from src.seed import set_seed
set_seed(SEED)

---
# Part 1: Data

The questions come from the authors' GitHub repo (github.com/changbingY/LingGym), which they built from
open-access grammar books (Language Science Press, CC BY 4.0). This downloads it at a fixed commit,
so the data can't change if the authors update their repo.

In [ ]:
if not os.path.exists(f"{RAW}/.git"):
    !git clone -q https://github.com/changbingY/LingGym.git {RAW}
!git -C {RAW} checkout -q {DATA_COMMIT}
print("LingGym data ready at", RAW, "| commit", DATA_COMMIT[:7])

### Load all 18 languages and check against the paper's Table 1

In [ ]:
import pandas as pd
from collections import Counter
from src.data import load_benchmark

records = load_benchmark(f"{RAW}/Benchmark_multiple_choice")
paper_table1 = {"Pichi": 2846, "Gyeli": 691, "Moloko": 439, "Fwe": 147, "Papuan Malay": 3766,
                "Rapa Nui": 1709, "Kagayanen": 550, "Vamale": 67, "Komnzo": 709, "Mauwake": 1787,
                "Kalamang": 656, "Ulwa": 1851, "Palula": 1674, "Tuatschin": 1113, "Japhug": 358,
                "Yauyos Quechua": 1143, "Mehweb": 85, "Ik": 21}
counts = Counter(r["language"] for r in records)
table = pd.DataFrame({"paper (Table 1)": paper_table1, "parsed by us": counts}).fillna(0).astype(int)
table["match"] = table["paper (Table 1)"] == table["parsed by us"]
print("Total parsed:", len(records), "| paper total: 19,612")
table.sort_values("paper (Table 1)", ascending=False)

The correct answer should be spread evenly over A–D (paper §4.2), so position alone can't give it away:

In [ ]:
pd.Series(Counter(r["answer"] for r in records)).sort_index().to_frame("questions").assign(
    percent=lambda d: (100 * d.questions / d.questions.sum()).round(1))

### One question at each hint level

S = sentence, G = gloss, KP = knowledge point (grammar rule), T = English translation.

In [ ]:
from src.prompts import build_prompt, MAIN_LEVELS

example = next(r for r in records if r["language"] == "Pichi" and len(r["knowledge_point"]) < 400)
for level in MAIN_LEVELS:
    print("=" * 30, level, "=" * 30)
    print(build_prompt(example, level), "\n")
print("Correct answer:", example["answer"], example["options"][example["answer"]])

### Does the answer leak through the hints?

The paper notes (§4.2) that the translation is *not* masked, so it may still hint at the answer.
We also check whether the hidden word survived masking in the knowledge point.

In [ ]:
def leaks(r, field):
    w = r["options"][r["answer"]]["word"].lower()
    return len(w) > 2 and w in r[field].lower()

leak = pd.DataFrame([{"language": r["language"], "word in KP": leaks(r, "knowledge_point"),
                      "word in translation": leaks(r, "translation")} for r in records])
leak_pct = (leak.groupby("language").mean() * 100).round(1).sort_values("word in KP", ascending=False)
leak_pct.to_csv("results/answer_leakage.csv")
leak_pct

### Our sample

All 19,612 questions × 4 levels would take many hours on a free GPU, so we take a **seeded random sample
of 150 questions per language** from 3 languages in 3 different families:

| Language | Family | Questions in benchmark |
|---|---|---|
| Pichi | Atlantic-Congo | 2,846 |
| Papuan Malay | Austronesian | 3,766 |
| Kalamang | Trans-New Guinea | 656 |

The paper reports accuracy per language (Appendix B), so we can compare against exactly these 3 languages.

In [ ]:
from src.data import sample_per_language, save_jsonl

chosen = [r for r in records if r["language_folder"] in LANGUAGES]
released = sample_per_language(chosen, N_PER_LANGUAGE, SEED)   # same sample every run
save_jsonl(released, "data/processed/released_sample.jsonl")
pd.Series(Counter(q["language"] for q in released)).to_frame("sampled questions")

---
# Part 2: Our own question generator (the "hacker" part)

Instead of only using the authors' questions, `src/question_gen.py` rebuilds question generation from
the paper's description (§4.2):

1. **Parse** the example sentences (sentence / gloss / translation) from the grammar-book data.
2. **Pick targets:** every word that is **bold** in the grammar (or contains a bold morpheme).
3. **Mask** the target in the sentence, gloss line and grammar explanation.
4. **Build 3 wrong options:**
   * **LCS (form):** the gloss sharing the *longest common substring* with the correct gloss
   * **Semantic:** the closest gloss by *Sentence-BERT* embedding similarity
   * **Chapter-local:** a random word–gloss pair from the same grammar chapter
5. **Shuffle** A–D with a fixed seed.

### Raw grammar data → clean example

In [ ]:
from src.question_gen import parse_igt_examples

csv_dir = f"{RAW}/CVS-format"
raw = pd.read_csv(f"{csv_dir}/Pichi/min_knowledge_points_05.csv")
print("RAW (LaTeX from the grammar book):\n", raw.loc[0, "Content 1"], "\n")

examples = parse_igt_examples(csv_dir, "Pichi")
ex = next(e for e in examples if any(e["bold"]))
print("PARSED:")
for k in ["words", "glosses", "bold", "translation"]:
    print(f"  {k:12}", ex[k])

### The wrong-option strategies, one at a time

In [ ]:
from src.question_gen import lcs_len, GlossEmbedder

correct = "3SG.SBJ"
for cand in ["3SG.OBJ", "3PL.SBJ", "1SG.SBJ", "maize", "DEF"]:
    print(f"LCS({correct!r}, {cand!r}) = {lcs_len(correct, cand)}")

In [ ]:
# Semantic neighbours: which glosses does Sentence-BERT think are closest?
glosses = sorted({g for e in examples for g in e["glosses"]})
emb = GlossEmbedder(glosses)          # downloads paraphrase-MiniLM-L6-v2 the first time
print("backend:", emb.backend)
for g in ["maize", "finger", "be.sweet"]:
    if g in emb.index:
        print(f"{g:10} ->", emb.ranked(g)[1:6])

### Generate questions for our 3 languages

We sample 150 bold targets per language. A target is skipped if we can't find 3 wrong options with
distinct words and glosses, so Papuan Malay and Kalamang end up with 149 questions (448 in total).

In [ ]:
from src.question_gen import generate_questions

own = []
for lang in LANGUAGES:
    qs = generate_questions(csv_dir, lang, seed=SEED, max_questions=N_PER_LANGUAGE)
    print(f"{lang}: {len(qs)} questions (similarity: {qs[0]['embedding_backend'] if qs else '-'})")
    own.extend(qs)
save_jsonl(own, "data/processed/own_questions.jsonl")
print(len(own), "questions in total\n")
q = own[0]
print(build_prompt(q, "S+G+KP+T"))
print("\nAnswer:", q["answer"], "| option types:", q["distractor_types"])

### How close is our generator to the authors'?

Number of target (bold) words we find vs. questions in the paper's Table 1. Differences come from the
authors' extra filters (e.g. they drop words containing "."); see README.

In [ ]:
rows = []
paper_counts = {"Pichi": 2846, "Papuan_Malay": 3766, "Kalamang": 656}
for lang in LANGUAGES:
    n_paper = paper_counts.get(lang)
    exs = parse_igt_examples(csv_dir, lang)
    rows.append({"language": lang, "paper questions": n_paper,
                 "our bold targets": sum(sum(e["bold"]) for e in exs),
                 "aligned examples": len(exs)})
counts_df = pd.DataFrame(rows)
counts_df.to_csv("results/generator_target_counts.csv", index=False)
counts_df

In [ ]:
# Sanity checks: answer positions balanced, every option distinct
print(Counter(q["answer"] for q in own))
assert all(len({o["word"] for o in q["options"].values()}) == 4 for q in own)
assert all(len({o["gloss"] for o in q["options"].values()}) == 4 for q in own)
print("All questions have 4 distinct words and 4 distinct glosses")

---
# Part 3: Model + live demo

Qwen2.5-7B-Instruct is the same model as the paper's Table 3. In float16 it needs ~15 GB, which doesn't fit
next to the activations on a 16 GB T4, so we load it in **4-bit (NF4)** (see `QUANTIZATION` in Settings). That's one source of difference from the paper.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from src.evaluate import load_model
model, tok = load_model(MODEL, quantization=QUANTIZATION)
MODEL_NAME = MODEL.split("/")[-1]
print("Loaded", MODEL_NAME, "on", model.device)

### Live demo: one question, four hint levels

How we read the model's answer: we look at the probability it gives to **A, B, C, D** as its very next
word. Watch the correct option's probability change as we add hints.

In [ ]:
import json
from src.evaluate import score_batch, letter_token_ids

letter_ids = letter_token_ids(tok)
demo_q = released[0]

print(build_prompt(demo_q, "S+G+KP+T"))
print("\nCorrect answer:", demo_q["answer"], "\n")
for level in MAIN_LEVELS:
    pred, probs = score_batch(model, tok, [build_prompt(demo_q, level)], letter_ids)[0]
    probs = json.loads(probs)
    bar = "█" * int(40 * probs[demo_q["answer"]])
    print(f"{level:10} predicted {pred} {'✓' if pred == demo_q['answer'] else '✗'}   "
          f"P(correct)={probs[demo_q['answer']]:.2f} {bar}")

---
# Part 4: Experiments

### Exp 0: Random baseline (should be ~25%)

In [ ]:
from src.experiment import run_experiment
_, acc0 = run_experiment("data/processed/released_sample.jsonl", "exp0_random", mode="random", seed=SEED)
acc0.round(1)

### Exp 1: our evaluator on the authors' released questions (reproduction)

In [ ]:
_, acc1 = run_experiment("data/processed/released_sample.jsonl", "exp1_released_qwen7b",
                         mode="score", model=model, tok=tok, model_name=MODEL_NAME,
                         seed=SEED, extra_config={"hf_model": MODEL, "quantization": QUANTIZATION})
acc1.round(1)

### Exp 2: same model on questions from our own generator (hacker part)

In [ ]:
_, acc2 = run_experiment("data/processed/own_questions.jsonl", "exp2_own_qwen7b",
                         mode="score", model=model, tok=tok, model_name=MODEL_NAME,
                         seed=SEED, extra_config={"hf_model": MODEL, "quantization": QUANTIZATION})
acc2.round(1)

### Exp 3: The paper's sampling setup

The paper samples a free-text reply (temperature 0.7, top-p 0.9, repetition penalty 1.1) and reads the
letter from it. We run that on 50 questions per language to measure how much our scoring shortcut matters.

In [ ]:
_, acc3 = run_experiment("data/processed/released_sample.jsonl", "exp3_released_qwen7b_sampling",
                         mode="generate", model=model, tok=tok, model_name=MODEL_NAME,
                         n_per_language=50, max_new_tokens=32,
                         seed=SEED, extra_config={"hf_model": MODEL, "quantization": QUANTIZATION})
acc3.round(1)   # 'unparsed' = replies with no A-D letter in them (counted as wrong)

---
# Part 5: Results vs. the paper

* **paper_matched**: the paper's per-language numbers (Appendix B) for *our* 3 languages, the fair comparison.
* **paper_all18**: the paper's headline Table 3 number over all 18 languages, for reference.
* **within_ci**: whether the paper's number falls inside our 95% bootstrap confidence interval.

In [ ]:
from src.analysis import compare_to_paper, plot_comparison

rel = pd.read_csv("results/exp1_released_qwen7b/predictions.csv")
own_pred = pd.read_csv("results/exp2_own_qwen7b/predictions.csv")
t1 = compare_to_paper(rel).assign(questions="released")
t2 = compare_to_paper(own_pred).assign(questions="own_generator")
comparison = pd.concat([t1, t2], ignore_index=True)
comparison = comparison[["questions"] + [c for c in comparison.columns if c != "questions"]]
comparison.to_csv("results/comparison.csv", index=False)

plot_comparison({"Paper": t1.set_index("level")["paper_matched"],
                 "Ours: released questions": t1.set_index("level")["ours"],
                 "Ours: own questions": t2.set_index("level")["ours"]},
                "figures/results_vs_paper.png",
                title=f"{MODEL.split('/')[-1]} on LingGym: paper vs. our reproduction")
comparison

In [ ]:
from IPython.display import Image
Image("figures/results_vs_paper.png")

In [ ]:
# Per-language breakdown vs. the paper's Appendix B, for both question sets
from src.analysis import paper_reference
per_lang_paper, _ = paper_reference("qwen2.5-7b")
rows = []
for name, run in [("released", "exp1_released_qwen7b"), ("own_generator", "exp2_own_qwen7b")]:
    ours = pd.read_csv(f"results/{run}/accuracy_by_language.csv").assign(questions=name)
    rows.append(ours.merge(per_lang_paper[["language", "level", "accuracy"]], on=["language", "level"],
                           suffixes=("_ours", "_paper")))
merged = pd.concat(rows, ignore_index=True)
merged["delta"] = merged.accuracy_ours - merged.accuracy_paper
merged = merged[["questions", "language", "level", "n", "accuracy_ours", "accuracy_paper", "delta"]].round(1)
merged.to_csv("results/per_language_vs_paper.csv", index=False)
merged

In [ ]:
# Scoring vs. sampling on the same questions (only if Exp 3 ran)
if os.path.exists("results/exp3_released_qwen7b_sampling/predictions.csv"):
    s = pd.read_csv("results/exp3_released_qwen7b_sampling/predictions.csv")
    sc = pd.read_csv("results/exp1_released_qwen7b/predictions.csv")
    sc = sc[sc.id.isin(s.id)]
    print(pd.DataFrame({"scoring (ours)": sc.groupby("level").correct.mean() * 100,
                        "sampling (paper)": s.groupby("level").correct.mean() * 100}).round(1))

### Exp 2: which wrong option fooled the model?

For every question from our generator that the model got wrong, look up which distractor type it picked.
If the LCS and Sentence-BERT options are picked more often than the random chapter one, they are doing
their job of being convincing.

In [ ]:
from src.data import load_jsonl
own_by_id = {q["id"]: q for q in load_jsonl("data/processed/own_questions.jsonl")}
wrong = own_pred[own_pred.correct == 0].copy()
wrong["picked"] = [own_by_id[i]["distractor_types"][p] for i, p in zip(wrong.id, wrong.pred)]
picked = pd.crosstab(wrong.level, wrong.picked, margins=True, margins_name="total")
picked.to_csv("results/exp2_own_qwen7b/wrong_answer_types.csv")
picked

---
# Part 6: Conclusion

On the authors' released questions (Exp 1) we reproduce the paper. At every hint level we are within
1.3 points of the paper's numbers for the same three languages, and the paper's number is inside our 95% CI
every time. Accuracy goes up with each hint (34, 40, 61, 76%), while random guessing stays at 22-27%.

Our own generator (Exp 2) makes harder questions: accuracy is 3.5 to 7.5 points below the paper. From S+G
upward the paper's number is outside our 95% CI, so the gap is probably real and not sampling noise. It is
biggest for Papuan Malay (10.4 points at S+G+KP+T). The wrong answers show why: the model was fooled by the
LCS (386) and Sentence-BERT (373) distractors about twice as often as by the random chapter one (191).

Exp 3 checks our main shortcut. Reading the A-D scores instead of sampling a reply changes accuracy by less
than 3 points on the 150-question subset, so it doesn't change any of the conclusions above.

Limitations: 450 questions from 3 of the 18 languages, one model, and 4-bit weights instead of fp16. We
also didn't test how much of the Exp 2 gap comes from our stricter masking of the knowledge point versus the
distractors themselves.

Challenges we hit along the way (details in the README):
- The 7B model doesn't fit a free 16 GB T4 in fp16, so we load it in 4-bit.
- Sampled replies are slow and random, so we score the A-D tokens instead (Exp 3 checks this).
- Colab disconnects mid-run, so predictions are saved after every batch and the run resumes.
- Each grammar marks the target word differently in LaTeX, so the parser handles both the morpheme and
  orthography lines, and a sanity check caught duplicate options (one word with two glosses).

### Save the exact library versions used in this session

In [ ]:
!pip freeze > results/colab_pip_freeze.txt
!cat results/exp1_released_qwen7b/environment.json

### Download your results (Colab)

Zips `results/`, `figures/` and `data/processed/` and downloads them from Colab.

In [ ]:
if IN_COLAB:
    !zip -qr results.zip results figures data/processed
    from google.colab import files
    files.download("results.zip")